# 1: Libraries

In [1]:
from mxnet import nd, gluon, init, autograd, gpu
from mxnet.gluon import nn
import mxnet as mx

import gc

import h5py
import numpy as np
import math
import random
import time
import datetime
from os import makedirs
import os

import pandas as pd
import scipy.signal
import scipy

# 2: Constants

In [2]:
# Путь к .mat файлу
path = './Data_large/'
# Имя .mat файла
signal_file_name = 'Signal_X_30sps_CDC_'
symbols_file_name = 'Symbols_1m_1ch_PR_'

number_of_files = 16

results_path = './Logs/'

gray_symbols_16qam = np.sqrt(0.1) * np.array(
                    [1+1j, 1+3j, 1-1j, 1-3j, 
                    3+1j, 3+3j, 3-1j, 3-3j, 
                    -1+1j, -1+3j, -1-1j, -1-3j,
                    -3+1j, -3+3j, -3-1j, -3-3j])

# Использование одинарной или двойной точности
swap_64_to_128_complex = False

if swap_64_to_128_complex:
    complex_t = np.complex128
    real_t = np.float64
else:
    complex_t = np.complex64
    real_t = np.float32

In [3]:
modulataion_order = 4           # 16-QAM
number_of_symbols = 524288
oversampling = 16                # number of samples per symbol interval
symbol_rate = 32.0e9
roll_off = 0.001                  # RRC filter roll-off factor
rrc_width = 1024                 # number of RRC filter coefficients
power_dbm = 1                   # per mode

number_of_spans = 20
span_length = 100e3
d_cd = 17e-6
alpha_db = 0.2
lambda_ = 1.55e-6
gamma = 1.4e-3 * 8/9
light_speed = 299792458

snr_cdc = 15.4303

In [4]:
number_of_bits = number_of_symbols * modulataion_order
signal_length = number_of_symbols * oversampling
power = 10 ** (power_dbm / 10) / 1000

fiber_length = number_of_spans * span_length
b_2 = -lambda_ ** 2 * d_cd / ( 2 * math.pi * light_speed)
alpha = alpha_db / 10 / np.log10(np.exp(1)) / 1000

In [5]:
# Переписать все

Fn = np.array([0])
fiberL = fiber_length
b2 = b_2
Fsym = symbol_rate

# 3: Load .mat file

In [6]:
# signal_rx_cdc = np.empty((number_of_files, signal_length), dtype=np.complex128)
symbols_tx = np.empty((number_of_files, number_of_symbols), dtype=np.complex128)
symbols_rx_file = np.empty((number_of_files, number_of_symbols), dtype=np.complex128)

for f in range(number_of_files):
    # df = pd.read_csv(path + signal_file_name + str(f + 1) + '.csv', header=None)
    # # signal_rx_real = df.to_numpy()
    # # signal_rx_cdc[f,:] = signal_rx_real[:,0] + 1j * signal_rx_real[:,1]
    
    df = pd.read_csv(path + symbols_file_name + str(f + 1) + '.csv', header=None)
    symbols_real = df.to_numpy()
    symbols_tx[f,:] = symbols_real[:,0] + 1j * symbols_real[:,1]
    symbols_rx_file[f,:] = symbols_real[:,2] + 1j * symbols_real[:,3]

# 4: Variables

# 4.1: Defined

In [7]:
# Параметры нейронной сети
ctx = gpu(3)                    # Расчеты на GPU
#ctx = mx.cpu(0)                # Расчеты на CPU

channel_numbers = [0]

train_portion = 0.8             # Часть данных, которая идет на обучение
CDC_data_size = 2 ** 21         # Количество символов, на которых обучаются CDC фильтры

num_step = 20                   # Количество шагов моделируемого DBP


manual_shift = True            # Учитывать временной сдвиг для разных каналов в коэффициентах FIR фильтров (False)
                                # или подбирать коэффицианты фильтров как для центральной частоты и потом просто cдвигать массивы (True)

epochs = 100000                  # Количество выполняемых эпох
batch_size = int(number_of_symbols / 16)             # Размер батча
lrate_start = 1e-3                    # Скорость обучения
lrate_CDC = 1e-3                # Скорость обучения CDC фильтров
lrate_stop = 1e-5
    
sym_filt = False # True
sym_nonlin = False # True

save_model_best = True             # Сохранять лучшую по loss модель в файл и затем использовать ее для тестирования
model_filename = "net.params"   # Имя файла, в который будет сохраняться модель

In [8]:
# Объединить функции 

In [9]:
%run CNN_DBP_functions_1.ipynb

In [10]:
# rrc = get_rrc_filter(oversampling, rrc_width, roll_off)

# new_oversampling = 1
# downsampling = int(oversampling / new_oversampling)
# signal_rx_cdc_downsampled = np.empty((number_of_files, int(signal_length / downsampling)), dtype=np.complex128)

# for f in range(number_of_files):
#     signal_rx_cdc_downsampled[f,:] = scipy.signal.fftconvolve(signal_rx_cdc[f,:], rrc, 'same')[1::oversampling]

# oversampling = new_oversampling
# signal_length = number_of_symbols * oversampling
# rrc = get_rrc_filter(oversampling, rrc_width, roll_off)

In [11]:
# signal_rx = np.empty((number_of_files, signal_length), dtype=np.complex128)

# for f in range(number_of_files):
#     signal_rx[f,:] = cd_operator(signal_rx_cdc_downsampled[f,:], fiber_length, oversampling, 0, 'f')

In [12]:
# signal_rx_nn = np.empty(number_of_files * signal_length, dtype=np.complex128)    # First iteration
symbols_tx_nn = np.empty(number_of_files * number_of_symbols, dtype=np.complex128) 
symbols_rx_nn = np.empty(number_of_files * number_of_symbols, dtype=np.complex128) 

for f in range(number_of_files):
    # signal_rx_nn[f*signal_length:(f+1)*signal_length] = signal_rx[f,:]
    symbols_tx_nn[f*number_of_symbols:(f+1)*number_of_symbols] = symbols_tx[f,:]
    symbols_rx_nn[f*number_of_symbols:(f+1)*number_of_symbols] = symbols_rx_file[f,:]

# 0.3.2 Calculated

In [13]:
# Вычисление вспомогательных параметров
channels = len(channel_numbers)
data_dimension = 2 * channels   # 1 polarization

central_frequency = 0

data_size = len(symbols_tx_nn)
train_data_size = int(data_size * train_portion)
# CDC_data_size = min(CDC_data_size, train_data_size)

nonlin_coef = gamma * power * 0.05 * 8/9

# 1.1 Data preparation

In [14]:
# Подготовка данных для нейронной сети
TX = np.empty((channels,data_size), dtype=complex_t)
RX = np.empty((channels,data_size), dtype=complex_t)

RX[0,:] = symbols_rx_nn # signal_rx_nn
RX[0,:] /= np.sqrt(np.mean(abs(RX[0,:]) ** 2))

TX[0,:] = symbols_tx_nn
TX[0,:] /= np.sqrt(np.mean(abs(TX[0,:]) ** 2))

X = np.empty((data_dimension, data_size), dtype=real_t)
y = np.empty((data_dimension, data_size), dtype=real_t)

X[::2] = np.real(RX[0,:])
X[1::2] = np.imag(RX[0,:])

y[::2] = np.real(TX[0,:])
y[1::2] = np.imag(TX[0,:])

del TX, RX

# 3 BER calculation

In [15]:
def train_full_statistic(lrate_start, X, y, run_count, filt_CDC_width_cur):
    filt_CDC_width = filt_CDC_width_cur            # Ширина CDC фильтра
    filt_CDC_last_width = 71        # Ширина CDC фильтра на последнем шаге
    filt_FD_width = 21              # Ширина FD фильтра

    nl_mem = 2                 # Количество соседних символов в каждом из направлений, используемых на нелинейном шаге
    nl_mem_nc1_1 = 10
    nl_mem_nc1_2 = 18
    nl_mem_nc2_1 = 1
    nl_mem_nc2_2 = 24
    nl_mem_nc3_1 = 1
    nl_mem_nc3_2 = 11
    
    filt_CDC_delay = int((filt_CDC_width-1)/2)
    filt_CDC_last_delay = int((filt_CDC_last_width-1)/2)
    filt_FD_delay = int((filt_FD_width-1)/2)

    if channels == 1:
        step_length, step_shifts, last_step_length, FD_step_length, step_output_shifts, full_output_shifts, max_full_shift = get_steps_shift(num_step, Fn, channel_numbers)
        last_shifts = [0, 0]  # Исправить, чтобы не было нужно
        last_output_shifts = [0, 0]  # Исправить, чтобы не было нужно
    else:
        step_length, step_shifts, last_step_length, last_shifts, last_output_shifts, FD_step_length, step_output_shifts, full_output_shifts, max_full_shift = get_steps_shift(num_step, Fn, channel_numbers)
        
        
    nl_memory_size, nl_shifts, max_nl_shift = get_nl_shift(channels, nl_mem, nl_mem_nc1_1, nl_mem_nc1_2, nl_mem_nc2_1, nl_mem_nc2_2, nl_mem_nc3_1, nl_mem_nc3_2, sym_nonlin)
        
    full_delay = num_step * (filt_CDC_delay + max_nl_shift)
    if last_step_length > 0:
        full_delay += filt_CDC_last_delay
    if FD_step_length > 0:
        full_delay += filt_FD_delay

    if sym_filt:
        filt_CDC_width = int((filt_CDC_width + 1) / 2)
        filt_CDC_last_width = int((filt_CDC_last_width + 1) / 2)
        
        
    dir_path = create_metadata(channels, channel_numbers, train_portion, train_data_size, CDC_data_size, num_step, filt_CDC_width, filt_CDC_last_width, filt_FD_width, nl_mem, nl_mem_nc1_1, nl_mem_nc1_2, nl_mem_nc2_1, nl_mem_nc2_2, nl_mem_nc3_1, nl_mem_nc3_2,
                   epochs, batch_size, lrate_start, lrate_CDC, manual_shift, sym_filt, sym_nonlin, save_model_best)
    
    X_train = X[:,2000:train_data_size+2000]
    y_train = y[:,2000:train_data_size+2000]
    
    
    #Вычисление дисперсионных фильтров
    # Исправить
    filt_CDC = get_CDCfilt_step_1ch(X_train[:2,:CDC_data_size], filt_CDC_width, step_length, step_shifts, step_output_shifts, dir_path, train_print=False)
    if last_step_length > 0:
        filt_CDC_last = get_CDCfilt_step(X_train[:2,:CDC_data_size], filt_CDC_last_width, last_step_length, last_shifts, last_output_shifts, dir_path, train_print=False)
        filt_CDC = filt_CDC + filt_CDC_last
    if FD_step_length > 0:
        filt_FD = get_FDfilt(X_train[:2,:CDC_data_size], filt_FD_width, FD_step_length, dir_path, train_print=False)
        filt_CDC = filt_CDC + filt_FD

    filt_CDC_JO = get_CDCfilt_JO_1ch(X_train[:2,:CDC_data_size], filt_CDC, filt_CDC_delay, step_length, last_step_length, filt_CDC_last_delay, FD_step_length, filt_FD_delay, step_shifts, last_shifts, step_output_shifts, 
                                 last_output_shifts, full_output_shifts, max_full_shift, dir_path, train_print=False)
    
    tensor_intra = nd.zeros((channels,1,nl_memory_size[0]))
    tensor_intra[:,:,nl_mem] = nd.ones((channels,1))

    if channels > 1:
        tensor_inter1 = 0.15 * nd.ones(((2*(channels-2)+2),1,nl_memory_size[1]))
    if channels == 3:
        tensor_inter2 = 0.08 * nd.ones((2,1,nl_memory_size[2]))
    if channels == 4:
        tensor_inter2 = 0.08 * nd.ones((4,1,nl_memory_size[2]))
        tensor_inter3 = 0.04 * nd.ones((2,1,nl_memory_size[3]))

    ber_min, ber_max, ber_sum, ber_sumsq = 1, 0, 0, 0
    bers = []
    epochs_sum = 0
    time_sum = 0
    
    if channels == 1:
        complexity = int(num_step * (0.5 * nl_memory_size[0] + 7 + 4 * filt_CDC_width)) # Переписать для одной пляризации

    if channels == 2:
        complexity = int(num_step * (0.5 * (nl_memory_size[0] + nl_memory_size[1]) + 7 + 4 * filt_CDC_width))

    if channels == 3:
        complexity = int(num_step * (0.5 * (nl_memory_size[0] + 4/3 * nl_memory_size[1] + 2/3 * nl_memory_size[2]) + 7 + 4 * filt_CDC_width))

    if channels == 4:
        complexity = int(num_step * (0.5 * (nl_memory_size[0] + 1.5 * nl_memory_size[1] + nl_memory_size[2] + 0.5 * nl_memory_size[3]) + 7 + 4 * filt_CDC_width))

    if last_step_length > 0:
        complexity += 4 * filt_CDC_last_width
    if FD_step_length > 0:
        complexity += 4 * filt_FD_width
    
    [train_dataloader, batch_train, number_of_bathes_train] = dataloader(X_train, y_train, batch_size, channels, data_dimension, full_delay, full_output_shifts, max_full_shift, True, pol=1)
    [all_data, batch_all, number_of_bathes_all] = dataloader(X, y, batch_size, channels, data_dimension, full_delay, full_output_shifts, max_full_shift, False, pol=1)
    
    for run in range(run_count):
        
        seed = int(datetime.datetime.now().timestamp()) # run
        mx.random.seed(seed)
        mx.random.seed(seed, ctx=mx.gpu(1))
        np.random.seed(seed)
        random.seed(seed)
    
        lrate = lrate_start
        
        net = gluon.nn.Sequential()
        with net.name_scope():
            for i in range(num_step):
                net.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[0][:,i:i+1,:], weightsIm=filt_CDC_JO[1][:,i:i+1,:], kernel_size=filt_CDC_width, shift=step_shifts, dimension=data_dimension, pol=1,enable_shift=manual_shift, sym=sym_filt))
                if channels == 1:
                    net.add(KerrActivationEnhanced_v2_1ch(dimension=data_dimension, tensor_intra=tensor_intra, nl_shifts=nl_shifts, nl_memory_size=nl_memory_size, max_nl_shift=max_nl_shift, 
                                                          nl_coef=nonlin_coef*step_length/channels, pol=1))
                if channels == 2:
                    # net.add(KerrActivationEnhanced_v2_2ch_SPM(dimension=data_dimension, tensor_intra=tensor_intra, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
                    # net.add(KerrActivationEnhanced_v2_2ch_LS_0(dimension=data_dimension, tensor_intra=tensor_intra, tensor_inter=tensor_inter1, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
                    net.add(KerrActivationEnhanced_v2_2ch(dimension=data_dimension, tensor_intra=tensor_intra, tensor_inter=tensor_inter1, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
                if channels == 3:
                    net.add(KerrActivationEnhanced_v2_3ch(dimension=data_dimension, tensor_inter1=tensor_inter1, tensor_inter2=tensor_inter2, tensor_intra=tensor_intra, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
                if channels == 4:
                    net.add(KerrActivationEnhanced_v2_4ch(dimension=data_dimension, tensor_inter1=tensor_inter1, tensor_inter2=tensor_inter2, tensor_inter3=tensor_inter3, tensor_intra=tensor_intra, 
                                                          nl_memory_size=nl_memory_size, nl_shifts=nl_shifts, max_nl_shift=max_nl_shift, nl_coef=nonlin_coef*step_length/channels))
            if last_step_length > 0:
                net.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[2], weightsIm=filt_CDC_JO[3], kernel_size=filt_CDC_last_width, shift=last_shifts, dimension=data_dimension, pol=1, enable_shift=manual_shift, sym=sym_filt))
            if FD_step_length > 0:
                net.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[len(filt_CDC_JO)-2], weightsIm=filt_CDC_JO[len(filt_CDC_JO)-1], kernel_size=filt_FD_width, dimension=data_dimension, pol=1, enable_shift=False))

        net.initialize(init=mx.init.Normal(sigma=0.05), ctx=ctx)

        trainer = gluon.Trainer(net.collect_params(), 'Adam', {'learning_rate': lrate})
        mse = gluon.loss.L2Loss()

        dir_name = dir_path                # Папка куда будет сохранен файл с моделью.
                                           # Если переменная пустая, то файл будет сохранен в папку, из которой запускается код

        filename = os.path.join(dir_name, model_filename)
        model_saved = False
        min_loss = 1

        decay_steps = 100
        steps_wo_min = 0
        demap_data_size = 2 ** 20
        start_train_time = time.time()
        for epoch in range(epochs):
            train_loss = nd.zeros(1, ctx=ctx)
            tic = time.time()
            for data, label in train_dataloader:
                data = data.as_in_context(ctx)
                label = label.as_in_context(ctx)
                with autograd.record():
                    output = net(data)
                    loss = mse(output, label)
                    loss = nd.mean(loss)

                loss.backward()
                trainer.step(1)
                train_loss += loss.mean().asscalar()
                loss.wait_to_read()

            # if (epoch + 1) % 10 == 0:
            #     print('epoch', epoch + 1, '-- loss', train_loss.asscalar()/number_of_bathes_train, '-- time', time.time()-tic)
            if min_loss > (train_loss.asscalar()/number_of_bathes_train):
                steps_wo_min = 0
                min_loss = train_loss.asscalar()/number_of_bathes_train
                if save_model_best:
                    net.save_parameters(filename)
                    model_saved = True
                    # print('epoch', epoch + 1, '-- Model saved', '-- loss', train_loss.asscalar()/number_of_bathes_train)
            else:
                steps_wo_min += 1

            if steps_wo_min >= decay_steps:
                steps_wo_min = 0
                lrate /= 2
                if lrate < lrate_stop:
                    break
                trainer.set_learning_rate(lrate)
                # print('epoch', epoch + 1, '-- Learning rate', lrate)

        output.wait_to_read()
        del data, label, output, train_loss
        gc.collect()
        
        train_time = time.time() - start_train_time
        
        # Тестирование нейронной сети, вычисление BER
        batch_trunc = batch_all - 2 * full_delay
        if manual_shift:
            batch_trunc -= max_full_shift
        y_pred = nd.empty([1, data_dimension, number_of_bathes_all * batch_trunc], dtype=real_t)
        y = nd.empty([1, data_dimension, number_of_bathes_all * batch_trunc], dtype=real_t)
        ### !!!!!!!!!!!!!!!!!!! Добавить симметричность

        if save_model_best and model_saved:
            netTest = gluon.nn.Sequential()
            with netTest.name_scope():
                for i in range(num_step):
                    netTest.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[0][:,i:i+1,:], weightsIm=filt_CDC_JO[1][:,i:i+1,:], kernel_size=filt_CDC_width, shift=step_shifts, dimension=data_dimension, pol=1, enable_shift=manual_shift, sym=sym_filt))
                    if channels == 1:
                        netTest.add(KerrActivationEnhanced_v2_1ch(dimension=data_dimension, tensor_intra=tensor_intra, nl_shifts=nl_shifts, nl_memory_size=nl_memory_size, max_nl_shift=max_nl_shift, 
                                                                  nl_coef=nonlin_coef*step_length/channels, pol=1))
                    if channels == 2:
                        # netTest.add(KerrActivationEnhanced_v2_2ch_SPM(dimension=data_dimension, tensor_intra=tensor_intra, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
                        netTest.add(KerrActivationEnhanced_v2_2ch_LS_0(dimension=data_dimension, tensor_intra=tensor_intra, tensor_inter=tensor_inter1, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
                        # netTest.add(KerrActivationEnhanced_v2_2ch(dimension=data_dimension, tensor_intra=tensor_intra, tensor_inter=tensor_inter1, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
                    if channels == 3:
                        netTest.add(KerrActivationEnhanced_v2_3ch(dimension=data_dimension, tensor_inter1=tensor_inter1, tensor_inter2=tensor_inter2, tensor_intra=tensor_intra, nl_shifts=nl_shifts, nl_coef=nonlin_coef*step_length/channels))
                    if channels == 4:
                        netTest.add(KerrActivationEnhanced_v2_4ch(dimension=data_dimension, tensor_inter1=tensor_inter1, tensor_inter2=tensor_inter2, tensor_inter3=tensor_inter3, tensor_intra=tensor_intra, 
                                                                  nl_memory_size=nl_memory_size, nl_shifts=nl_shifts, max_nl_shift=max_nl_shift, nl_coef=nonlin_coef*step_length/channels))
                if last_step_length > 0:
                    netTest.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[2], weightsIm=filt_CDC_JO[3], kernel_size=filt_CDC_last_width, shift=last_shifts, dimension=data_dimension, pol=1, enable_shift=manual_shift, sym=sym_filt))
                if FD_step_length > 0:
                    netTest.add(ComplexConvPredefined(weightsRe=filt_CDC_JO[len(filt_CDC_JO)-2], weightsIm=filt_CDC_JO[len(filt_CDC_JO)-1], kernel_size=filt_FD_width, dimension=data_dimension, pol=1, enable_shift=False))

            netTest.load_parameters(filename, ctx=ctx)

        for batch_idx, data in enumerate(all_data):
            data[0] = data[0].as_in_context(ctx)
            if save_model_best and model_saved:
                output = netTest(data[0])
            else:
                output = net(data[0])
            y_pred[:,:,batch_idx*batch_trunc:(batch_idx+1)*batch_trunc] = output.as_in_context(mx.cpu(0))
            y[:,:,batch_idx*batch_trunc:(batch_idx+1)*batch_trunc] = data[1]

        output.wait_to_read() 
        y_pred = y_pred.asnumpy()
        y = y.asnumpy()

        demap_data_size = 2 ** 20
        mean_BER_NN = 0
        # mean_BER_NN_phase = 0
        for i in range(int(data_dimension/2)):
            a = demapper(y[:,2*i:2*(i+1),10000:10000+demap_data_size], y_pred[:,2*i:2*(i+1),10000:10000+demap_data_size])
            mean_BER_NN += calculate_ber(y_pred[:,2*i:2*(i+1),:] * a, y[:,2*i:2*(i+1),:])
            
            # tx_s = y[:,2*i:2*(i+1),:]
            # rx_s = y_pred[:,2*i:2*(i+1),:] * a
            # mean_BER_NN_phase += calculate_ber_from_symbols(tx_s[0,0,:] + 1j * tx_s[0,1,:], demodulate_signal_phase_1sps(rx_s[0,0,:] + 1j * rx_s[0,1,:], tx_s[0,0,:] + 1j * tx_s[0,1,:])[0])

        #mean_BER_CDC = (np.mean(BER_CDC_X[channel_numbers]) + np.mean(BER_CDC_Y[channel_numbers]))/2
        #mean_BER_DBP = (np.mean(BER_DBP_X[channel_numbers]) + np.mean(BER_DBP_Y[channel_numbers]))/2
        mean_BER_NN /= data_dimension/2
        # mean_BER_NN_phase /= data_dimension/2
        
        # print('BER phase:', mean_BER_NN_phase)
        
        ber = mean_BER_NN
        ber_min = min(ber_min, ber)
        ber_max = max(ber_max, ber)
        ber_sum += ber
        ber_sumsq += ber * ber
        bers.append(ber)
        ber_med = np.median(bers)

        epochs_sum += epoch + 1
        time_sum += train_time
        
        q_mean = ber_to_qfactor(ber_sum / (run + 1))
        dq_mean = q_mean - snr_cdc
        
        #if run == 0:
        #    print('BER CDC: ', mean_BER_CDC)
        #    print('BER DBP Huawei: ', mean_BER_DBP)
        #    print()
        
        print("Run: {} -- min loss: {:.6f}   epochs: {}   mean epochs: {}   time: {:d}   mean time: {:d} -- BER: {:.7f} -- BER min: {:.7f}   BER max: {:.7f}   mean: {:.7f}   min/max: {:.3f}   Q: {:.4f}   dQ: {:.4f}".format(run + 1, min_loss, 
                epoch + 1, int(epochs_sum / (run + 1)), int(train_time / 60), int(time_sum / (run + 1) / 60), mean_BER_NN, ber_min, ber_max, ber_sum / (run + 1), ber_min / ber_max, q_mean, dq_mean))
        
        f = open(dir_path + 'metadata.md', 'a')
        
        if run == 0:
            f.write('\n')
            f.write('Epochs to decay: {}\n'.format(decay_steps))
            # f.write('\nBER CDC: {}\n'.format(mean_BER_CDC))
            # f.write('BER DBP Huawei: {}\n'.format(mean_BER_DBP))
            f.write('\n')

        f.write('\nRun: {} -- min loss: {:.6f}   epochs: {}   mean epochs: {}   time: {:d}   mean time: {:d} -- BER: {:.7f} -- BER min: {:.7f}   BER max: {:.7f}   mean: {:.7f}   min/max: {:.3f}   Q: {:.4f}   dQ: {:.4f}"\n'.format(run + 1, 
                min_loss, epoch + 1, int(epochs_sum / (run + 1)), int(train_time / 60), int(time_sum / (run + 1) / 60), mean_BER_NN, ber_min, ber_max, ber_sum / (run + 1), ber_min / ber_max, q_mean, dq_mean))

        # Переделать сложность для нейлинейности на последних слоях, нелинейность для 1 поляризации
        if run == run_count - 1:
            f.write('\n\n')
            f.write('\nComplexity: {}\n'.format(complexity))
            f.write('\n\n')
            f.write('To Overleaf   {} & {:.7f} & {:.4f} & {:.4f}\n'.format(filt_CDC_width, ber_sum / (run + 1), q_mean, dq_mean))
    
            print()
            print('Complexity: ', complexity)

        f.close()

In [ ]:
run_count = 3

# train_full_statistic(lrate_start, X, y, run_count)

# for filt_CDC_width_cur in [51, 61, 81, 101, 121, 151]:
for filt_CDC_width_cur in [151, 201]:
    print(filt_CDC_width_cur)
    train_full_statistic(lrate_start, X, y, run_count, filt_CDC_width_cur)

151


[15:21:09] src/operator/nn/./cudnn/./cudnn_algoreg-inl.h:97: Running performance tests to find the best convolution algorithm, this can take a while... (setting env variable MXNET_CUDNN_AUTOTUNE_DEFAULT to 0 to disable)
[15:42:10] src/operator/nn/./cudnn/./cudnn_algoreg-inl.h:97: Running performance tests to find the best convolution algorithm, this can take a while... (setting env variable MXNET_CUDNN_AUTOTUNE_DEFAULT to 0 to disable)
